# Vesuvius Challenge 2025 - 3-Model Ensemble Inference

Ensembles three models with different architectures, folds, and training recipes:
- **V11 Fold 0**: TopoPreservingUNet3D (10M params) — [320,320] bottleneck, trained 192³, Dice 0.5793
- **V11.1 Fold 1**: TopoPreservingUNet3D (18M params) — [512,512] bottleneck, trained 128³, Dice 0.5596
- **V2**: ResEncUNet3D + scSE (112M params) — skeleton/clDice losses, simple Z-score normalization

## Why 3 Models?
From Kaggle discussion analysis:
1. **VOI metric includes background** → ensemble averaging smooths boundaries, improving VOI
2. **Ignore mask creates artificial holes** (0→1 hole costs ~0.05 on LB!) → more models = smoother = fewer spurious holes
3. **Topo score very sensitive** → ensemble diversity reduces topology errors
4. **Different folds + architectures** → maximum diversity for robust predictions

## Ensemble Strategy
- V11-F0 (40%): strongest individual model, fold 0 validation
- V11.1-F1 (25%): different fold + architecture, adds diversity
- V2 (35%): completely different architecture, maximum diversity
- Optional: SWA (weight averaging) of fold 0 periodic checkpoints

## OOM Fixes (Conv3d im2col workspace)
- V11/V11.1 SurfaceRefinementBlock: tiled (TILE_DEPTH=32)
- V2 Encoder stage 0 + Decoder last stage: tiled (TILE_DEPTH=32)

## Post-Processing (Discussion-Informed)
- 3D binary closing → 3D hole filling → slicewise morphology
- Topology-safe: revert any operation that merges components
- Aggressive hole filling: betti-1 errors cost ~0.05 per hole on LB

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import gc
import sys
import traceback
import warnings
import zipfile
from pathlib import Path
from typing import List, Tuple
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import tifffile
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from scipy.ndimage import (
    binary_fill_holes, binary_closing, binary_opening,
    label, generate_binary_structure
)
from skimage.morphology import remove_small_objects

warnings.filterwarnings('ignore')

# =============================================================================
# GPU DETECTION
# =============================================================================
N_GPUS = torch.cuda.device_count()
print(f"Available GPUs: {N_GPUS}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

for i in range(N_GPUS):
    props = torch.cuda.get_device_properties(i)
    total_mem = props.total_memory / 1e9
    try:
        free_mem = torch.cuda.mem_get_info(i)[0] / 1e9
        print(f"  GPU {i}: {props.name} ({total_mem:.1f} GB total, {free_mem:.1f} GB free)")
    except:
        print(f"  GPU {i}: {props.name} ({total_mem:.1f} GB total)")

DEVICE = "cuda" if N_GPUS > 0 else "cpu"
print(f"Using device: {DEVICE}")

# =============================================================================
# CONFIGURATION — 3-MODEL ENSEMBLE
# =============================================================================
@dataclass
class Config:
    # Paths
    TEST_ROOT: Path = Path("/kaggle/input/vesuvius-challenge-surface-detection/test_images")
    TEST_CSV: Path = Path("/kaggle/input/vesuvius-challenge-surface-detection/test.csv")
    OUTPUT_DIR: Path = Path("/kaggle/working")

    # =========================================================================
    # MODEL 1: V11 Fold 0 (old architecture — strongest individual, Dice 0.5793)
    # Features [32,64,128,256,320,320], Blocks [1,2,3,4,6,6], trained on 192³
    # =========================================================================
    V11_F0_CHECKPOINT: Path = Path(
        "/kaggle/input/models/manish756/v11-vesuvius-model/pytorch/default/6/"
        "checkpoints_v11/fold_0/best_model.pth"
    )
    # Optional: extra fold 0 checkpoints for SWA weight averaging
    # e.g., [Path("...checkpoint_epoch_300.pth"), Path("...checkpoint_epoch_600.pth")]
    V11_F0_EXTRA_CHECKPOINTS: List[Path] = field(default_factory=list)
    V11_F0_FEATURES: List[int] = field(default_factory=lambda: [32, 64, 128, 256, 320, 320])
    V11_F0_N_BLOCKS: List[int] = field(default_factory=lambda: [1, 2, 3, 4, 6, 6])

    # =========================================================================
    # MODEL 2: V11.1 Fold 1 (new architecture — 512ch bottleneck, Dice 0.5596)
    # Features [32,64,128,256,512,512], Blocks [1,2,3,4,4,4], trained on 128³
    # Different fold → different validation volumes → adds ensemble diversity
    # =========================================================================
    V11_F1_CHECKPOINT: Path = Path(
        "/kaggle/input/models/manish756/v11-vesuvius-model/pytorch/default/7/"
        "checkpoints_v11/fold_1/best_model.pth"
    )
    V11_F1_FEATURES: List[int] = field(default_factory=lambda: [32, 64, 128, 256, 512, 512])
    V11_F1_N_BLOCKS: List[int] = field(default_factory=lambda: [1, 2, 3, 4, 4, 4])

    # =========================================================================
    # MODEL 3: V2 (ResEncUNet3D + scSE — completely different architecture)
    # =========================================================================
    V2_CHECKPOINT: Path = Path(
        "/kaggle/input/vesuvius-esembled-model/pytorch/default/1/"
        "best_model_Score_05315.pth"
    )
    V2_FEATURES: List[int] = field(default_factory=lambda: [32, 64, 128, 256, 320, 320])
    V2_N_BLOCKS: List[int] = field(default_factory=lambda: [1, 3, 4, 6, 6, 6])

    # =========================================================================
    # INFERENCE SETTINGS
    # All models use 192³ patches for inference (fully convolutional — works
    # even for V11.1 trained on 128³, gives more context)
    # =========================================================================
    PATCH_SIZE: Tuple[int, int, int] = (192, 192, 192)
    OVERLAP: float = 0.7
    USE_AMP: bool = True

    # =========================================================================
    # ENSEMBLE WEIGHTS (sum to 1.0)
    # V11-F0: strongest Dice, fold 0
    # V11.1-F1: different fold + architecture, adds diversity
    # V2: maximum architectural diversity
    # =========================================================================
    V11_F0_WEIGHT: float = 0.40
    V11_F1_WEIGHT: float = 0.25
    V2_WEIGHT: float = 0.35

    # =========================================================================
    # POST-PROCESSING (improved based on Kaggle discussion analysis)
    # - 3D closing + hole fill catches topology errors slicewise misses
    # - betti-1 errors (holes) cost ~0.05 per hole on LB
    # =========================================================================
    THRESHOLD: float = 0.50
    MIN_COMPONENT_SIZE: int = 50

    # =========================================================================
    # TILING — controls depth-wise tiling for Conv3d at full resolution
    # =========================================================================
    TILE_DEPTH: int = 32

    DEVICE: str = "cuda"

cfg = Config()

# Validate weights sum to 1.0
w_sum = cfg.V11_F0_WEIGHT + cfg.V11_F1_WEIGHT + cfg.V2_WEIGHT
assert abs(w_sum - 1.0) < 1e-6, f"Ensemble weights must sum to 1.0, got {w_sum}"

print(f"\n3-Model Ensemble Configuration:")
print(f"  V11-F0: {cfg.V11_F0_CHECKPOINT}")
print(f"    Features: {cfg.V11_F0_FEATURES}, Blocks: {cfg.V11_F0_N_BLOCKS}")
print(f"    Weight: {cfg.V11_F0_WEIGHT}, Extra checkpoints: {len(cfg.V11_F0_EXTRA_CHECKPOINTS)}")
print(f"  V11.1-F1: {cfg.V11_F1_CHECKPOINT}")
print(f"    Features: {cfg.V11_F1_FEATURES}, Blocks: {cfg.V11_F1_N_BLOCKS}")
print(f"    Weight: {cfg.V11_F1_WEIGHT}")
print(f"  V2: {cfg.V2_CHECKPOINT}")
print(f"    Weight: {cfg.V2_WEIGHT}")
print(f"  Patch: {cfg.PATCH_SIZE}, Overlap: {cfg.OVERLAP}")
print(f"  Threshold: {cfg.THRESHOLD}")
print(f"  Tile depth: {cfg.TILE_DEPTH}")
print(f"  Device: {DEVICE}")

# Clear GPU memory
for i in range(N_GPUS):
    with torch.cuda.device(i):
        torch.cuda.empty_cache()
gc.collect()
print("GPU memory cleared")

In [ ]:
# =============================================================================
# MODEL 1: TopoPreservingUNet3D (V11)
# =============================================================================

def get_num_groups(channels, max_groups=32):
    for g in [max_groups, 16, 8, 4, 2, 1]:
        if channels % g == 0:
            return g
    return 1


class HybridConv3d(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        mid_ch = out_ch // 2
        self.conv_xy = nn.Conv3d(in_ch, mid_ch, kernel_size=(1, 3, 3), padding=(0, 1, 1), bias=False)
        self.conv_z = nn.Conv3d(in_ch, out_ch - mid_ch, kernel_size=(3, 1, 1), padding=(1, 0, 0), bias=False)
        self.norm = nn.GroupNorm(get_num_groups(out_ch), out_ch)
        self.act = nn.LeakyReLU(0.01, inplace=True)

    def forward(self, x):
        return self.act(self.norm(torch.cat([self.conv_xy(x), self.conv_z(x)], dim=1)))


class V11ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, use_hybrid=False):
        super().__init__()
        if use_hybrid:
            self.conv = HybridConv3d(in_ch, out_ch)
        else:
            self.conv = nn.Sequential(
                nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False),
                nn.GroupNorm(get_num_groups(out_ch), out_ch),
                nn.LeakyReLU(0.01, inplace=True),
            )

    def forward(self, x):
        return self.conv(x)


class MultiScaleResBlock(nn.Module):
    def __init__(self, channels, scales=2, use_hybrid=False):
        super().__init__()
        self.scales = scales
        self.width = channels // scales
        self.convs = nn.ModuleList(
            [V11ConvBlock(self.width, self.width, use_hybrid) for _ in range(scales - 1)]
        )
        self.norm = nn.GroupNorm(get_num_groups(channels), channels)

    def forward(self, x):
        splits = torch.chunk(x, self.scales, dim=1)
        outputs = [splits[0]]
        for i, conv in enumerate(self.convs):
            out = conv(splits[i + 1] + outputs[-1]) if i > 0 else conv(splits[i + 1])
            outputs.append(out)
        return x + self.norm(torch.cat(outputs, dim=1))


class AttentionBlock(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.gap = nn.AdaptiveAvgPool3d(1)
        self.fc1 = nn.Linear(channels, max(channels // reduction, 8))
        self.fc2 = nn.Linear(max(channels // reduction, 8), channels)
        self.conv_spatial = nn.Conv3d(2, 1, kernel_size=7, padding=3, bias=False)

    def forward(self, x):
        b, c = x.shape[:2]
        ca = torch.sigmoid(self.fc2(F.relu(self.fc1(self.gap(x).view(b, c))))).view(b, c, 1, 1, 1)
        x_ca = x * ca
        sa = torch.sigmoid(
            self.conv_spatial(
                torch.cat([x_ca.mean(1, keepdim=True), x_ca.max(1, keepdim=True)[0]], 1)
            )
        )
        return x_ca * sa


class SurfaceRefinementBlock(nn.Module):
    """
    OOM FIX: Conv3d(128->32, 3x3x3) on full 192^3 needs 22.78 GiB workspace.
    Workspace scales as D/192 * 22.78 GiB.

    TILE_DEPTH=32 -> workspace ~3.8 GiB per conv layer.
    With model (~0.02 GB) + encoder activations (~2.5 GB) + workspace (~3.8 GB)
    = ~6.3 GB peak, fits comfortably in T4 15 GB (even with DataParallel replicas).
    
    Single pass: edge_conv and refine run on same tile sequentially.
    Conv workspace is freed between layers so they don't stack.
    """
    TILE_DEPTH = 32

    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.edge_conv = nn.Conv3d(in_ch, in_ch, 3, padding=1, bias=False)
        self.refine = nn.Sequential(
            nn.Conv3d(in_ch * 2, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(get_num_groups(out_ch), out_ch),
            nn.LeakyReLU(0.01, inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(get_num_groups(out_ch), out_ch),
            nn.LeakyReLU(0.01, inplace=True),
        )

    def _forward_tiled(self, x):
        """Process one sample via depth-wise tiling to avoid OOM."""
        B, C, D, H, W = x.shape
        assert B == 1
        tile = self.TILE_DEPTH
        pad = 1  # 3x3x3 conv boundary overlap

        output_slices = []
        for d_start in range(0, D, tile):
            d_end = min(d_start + tile, D)
            # Expand for conv overlap
            d_start_pad = max(0, d_start - pad)
            d_end_pad = min(D, d_end + pad)

            x_tile = x[:, :, d_start_pad:d_end_pad, :, :]
            edge_tile = torch.abs(self.edge_conv(x_tile))
            cat_tile = torch.cat([x_tile, edge_tile], dim=1)
            del x_tile, edge_tile

            out_tile = self.refine(cat_tile)
            del cat_tile

            # Crop to valid region
            valid_start = d_start - d_start_pad
            valid_end = valid_start + (d_end - d_start)
            output_slices.append(out_tile[:, :, valid_start:valid_end, :, :])
            del out_tile

        return torch.cat(output_slices, dim=2)

    def forward(self, x):
        B = x.shape[0]
        D = x.shape[2]
        # Small inputs: no tiling needed
        if D <= self.TILE_DEPTH:
            return self.refine(torch.cat([x, torch.abs(self.edge_conv(x))], dim=1))
        # Large inputs: tile each sample
        outputs = []
        for i in range(B):
            outputs.append(self._forward_tiled(x[i:i+1]))
        return torch.cat(outputs, dim=0)


class TopoPreservingUNet3D(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, features=None, n_blocks=None,
                 use_attention=True, use_hybrid_conv=True, use_surface_refinement=True):
        super().__init__()
        features = features or [32, 64, 128, 256, 320, 320]
        n_blocks = n_blocks or [1, 2, 3, 4, 6, 6]
        self.features = features

        self.encoders = nn.ModuleList()
        self.attentions = nn.ModuleList()
        self.pools = nn.ModuleList()

        for i, (feat, nb) in enumerate(zip(features, n_blocks)):
            in_channels = in_ch if i == 0 else features[i - 1]
            layers = [V11ConvBlock(in_channels, feat, use_hybrid=use_hybrid_conv and i > 0)]
            layers.extend([MultiScaleResBlock(feat, 2, use_hybrid_conv) for _ in range(nb)])
            self.encoders.append(nn.Sequential(*layers))
            self.attentions.append(
                AttentionBlock(feat) if use_attention and i >= 2 else nn.Identity()
            )
            if i < len(features) - 1:
                self.pools.append(nn.Conv3d(feat, feat, 2, stride=2))

        self.ups = nn.ModuleList()
        self.dec_convs = nn.ModuleList()
        for i in range(len(features) - 2, -1, -1):
            self.ups.append(nn.ConvTranspose3d(features[i + 1], features[i], 2, stride=2))
            if use_surface_refinement and i == 0:
                self.dec_convs.append(SurfaceRefinementBlock(features[i] * 2, features[i]))
            else:
                self.dec_convs.append(
                    nn.Sequential(
                        V11ConvBlock(features[i] * 2, features[i], use_hybrid_conv),
                        MultiScaleResBlock(features[i], 2, use_hybrid_conv),
                    )
                )
        self.final = nn.Conv3d(features[0], out_ch, 1)

    def forward(self, x):
        enc_features = []
        for i, (enc, att) in enumerate(zip(self.encoders, self.attentions)):
            x = att(enc(x))
            enc_features.append(x)
            if i < len(self.pools):
                x = self.pools[i](x)
        enc_features = enc_features[::-1]
        x = enc_features[0]
        for i, (up, dec) in enumerate(zip(self.ups, self.dec_convs)):
            x = up(x)
            skip = enc_features[i + 1]
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:], mode='trilinear', align_corners=False)
            x = dec(torch.cat([x, skip], dim=1))
        return self.final(x)


print("V11 model (TopoPreservingUNet3D) defined")
print("  SurfaceRefinementBlock: depth-tiled (32 slices, ~3.8 GiB workspace)")

In [ ]:
# =============================================================================
# MODEL 2: ResEncUNet3D + scSE (V2)
# =============================================================================
# OOM FIX: Conv3d at full 192^3 resolution causes massive im2col workspace:
#   Conv3d(C_in, _, 3x3x3) on (1, C_in, 192, 192, 192)
#   workspace = C_in * 27 * 192^3 * 2 bytes
#   C_in=32: 11.39 GiB, C_in=64: 22.78 GiB — both exceed T4 15 GB
#
# AFFECTED LAYERS IN V2 (at full 192^3 resolution):
# 1. Encoder stage 0: V2ConvBlock(1, 32) → Conv3d(1, 32, 3) → 0.37 GiB (OK)
#                      ResBlock(32) → Conv3d(32, 32, 3) → 11.39 GiB (OOM!)
# 2. Decoder last stage: V2ConvBlock(64, 32) → Conv3d(64, 32, 3) → 22.78 GiB (OOM!)
#
# SOLUTION: Tile BOTH encoder stage 0 AND decoder last stage in depth.
# =============================================================================

TILE_DEPTH = 32  # Shared tile depth for both models


class V2ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, num_groups=8):
        super().__init__()
        num_groups = min(num_groups, out_ch)
        while out_ch % num_groups != 0 and num_groups > 1:
            num_groups -= 1
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(num_groups, out_ch),
            nn.LeakyReLU(0.01, inplace=True),
        )

    def forward(self, x):
        return self.conv(x)


class ResBlock(nn.Module):
    def __init__(self, channels, n_convs=2):
        super().__init__()
        self.blocks = nn.Sequential(*[V2ConvBlock(channels, channels) for _ in range(n_convs)])

    def forward(self, x):
        return x + self.blocks(x)


class scSEBlock(nn.Module):
    def __init__(self, channels, reduction=2):
        super().__init__()
        self.cse_pool = nn.AdaptiveAvgPool3d(1)
        self.cse_fc1 = nn.Linear(channels, channels // reduction)
        self.cse_fc2 = nn.Linear(channels // reduction, channels)
        self.sse_conv = nn.Conv3d(channels, 1, 1)

    def forward(self, x):
        b, c = x.shape[:2]
        cse = torch.sigmoid(
            self.cse_fc2(F.relu(self.cse_fc1(self.cse_pool(x).view(b, c))))
        ).view(b, c, 1, 1, 1)
        sse = torch.sigmoid(self.sse_conv(x))
        return x * cse + x * sse


def _tiled_forward(module, x, tile_depth=32):
    """
    Run ANY nn.Module on large spatial tensors via depth tiling.
    Handles Conv3d workspace reduction from ~11-23 GiB to ~0.6-1.9 GiB.
    
    Works for both Sequential blocks and single modules.
    Adds 1-voxel padding overlap for 3x3x3 conv boundary correctness.
    """
    B, C, D, H, W = x.shape
    if D <= tile_depth:
        return module(x)

    pad = 1  # 3x3x3 conv boundary overlap
    output_slices = []
    for d_start in range(0, D, tile_depth):
        d_end = min(d_start + tile_depth, D)
        d_start_pad = max(0, d_start - pad)
        d_end_pad = min(D, d_end + pad)

        x_tile = x[:, :, d_start_pad:d_end_pad, :, :]
        out_tile = module(x_tile)
        del x_tile

        valid_start = d_start - d_start_pad
        valid_end = valid_start + (d_end - d_start)
        output_slices.append(out_tile[:, :, valid_start:valid_end, :, :])
        del out_tile

    return torch.cat(output_slices, dim=2)


def _tiled_resblock(resblock, x, tile_depth=32):
    """
    Run a ResBlock with depth tiling.
    ResBlock = x + blocks(x), so we tile blocks(x) and add the residual.
    
    This is needed because ResBlock.blocks contains Conv3d(32,32,3) at 192^3
    which needs 11.39 GiB workspace.
    """
    D = x.shape[2]
    if D <= tile_depth:
        return resblock(x)

    # Tile only the conv blocks, then add residual
    pad = 2  # ResBlock has 2 sequential Conv3d(3x3x3), need 2-voxel overlap
    B, C, D, H, W = x.shape
    output_slices = []
    for d_start in range(0, D, tile_depth):
        d_end = min(d_start + tile_depth, D)
        d_start_pad = max(0, d_start - pad)
        d_end_pad = min(D, d_end + pad)

        x_tile = x[:, :, d_start_pad:d_end_pad, :, :]
        # resblock.blocks is the Sequential of V2ConvBlocks
        conv_out = resblock.blocks(x_tile)
        # Add residual
        out_tile = x_tile + conv_out
        del conv_out

        valid_start = d_start - d_start_pad
        valid_end = valid_start + (d_end - d_start)
        output_slices.append(out_tile[:, :, valid_start:valid_end, :, :])
        del x_tile, out_tile

    return torch.cat(output_slices, dim=2)


class ResEncUNet3D(nn.Module):
    """V2 model - exact architecture from training notebook for weight compatibility.
    
    OOM-safe: tiles Conv3d operations at full resolution (encoder stage 0 + decoder last stage).
    """

    def __init__(self, in_ch=1, out_ch=1, features=None, n_blocks=None,
                 use_scse=True, use_deep_supervision=True):
        super().__init__()
        if features is None:
            features = [32, 64, 128, 256, 320, 320]
        if n_blocks is None:
            n_blocks = [1, 3, 4, 6, 6, 6]

        self.features = features
        self.use_scse = use_scse
        self.use_deep_supervision = use_deep_supervision

        # Encoder
        self.encoders = nn.ModuleList()
        self.attentions = nn.ModuleList()
        self.pools = nn.ModuleList()

        for i, (feat, nb) in enumerate(zip(features, n_blocks)):
            in_channels = in_ch if i == 0 else features[i - 1]
            encoder = nn.Sequential(
                V2ConvBlock(in_channels, feat),
                *[ResBlock(feat) for _ in range(nb)],
            )
            self.encoders.append(encoder)
            self.attentions.append(scSEBlock(feat) if use_scse else nn.Identity())
            if i < len(features) - 1:
                self.pools.append(nn.Conv3d(feat, feat, 2, 2))

        # Decoder
        self.ups = nn.ModuleList()
        self.dec_convs = nn.ModuleList()
        for i in range(len(features) - 2, -1, -1):
            self.ups.append(nn.ConvTranspose3d(features[i + 1], features[i], 2, 2))
            self.dec_convs.append(V2ConvBlock(features[i] * 2, features[i]))

        self.final = nn.Conv3d(features[0], out_ch, 1)

        # Deep supervision heads (needed for weight loading)
        if use_deep_supervision:
            self.ds_heads = nn.ModuleList()
            n_ds_outputs = min(3, len(features) - 1)
            for i in range(n_ds_outputs):
                ds_in_channels = features[-(i + 2)]
                self.ds_heads.append(nn.Conv3d(ds_in_channels, out_ch, 1))

    def forward(self, x):
        enc_features = []
        for i, (enc, att) in enumerate(zip(self.encoders, self.attentions)):
            if i == 0:
                # =============================================================
                # ENCODER STAGE 0: runs at full 192^3 resolution
                # V2ConvBlock(1, 32) → Conv3d(1, 32, 3) → 0.37 GiB (OK)
                # ResBlock(32) → Conv3d(32, 32, 3) → 11.39 GiB (OOM!)
                # Solution: run initial conv normally, tile each ResBlock
                # =============================================================
                # enc is Sequential(V2ConvBlock, ResBlock, ...)
                # Run V2ConvBlock (first layer) — Conv3d(1, 32, 3) is small (0.37 GiB)
                x = enc[0](x)
                # Run each ResBlock with tiling
                for j in range(1, len(enc)):
                    x = _tiled_resblock(enc[j], x, tile_depth=TILE_DEPTH)
            else:
                x = enc(x)

            x = att(x)
            enc_features.append(x)
            if i < len(self.pools):
                x = self.pools[i](x)

        enc_features = enc_features[::-1]
        x = enc_features[0]
        n_dec = len(self.ups)

        for i, (up, dec) in enumerate(zip(self.ups, self.dec_convs)):
            x = up(x)
            skip = enc_features[i + 1]
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:], mode='trilinear', align_corners=False)
            x = torch.cat([x, skip], dim=1)

            if i == n_dec - 1:
                # =============================================================
                # DECODER LAST STAGE: runs at full 192^3 resolution
                # V2ConvBlock(64, 32) → Conv3d(64, 32, 3) → 22.78 GiB (OOM!)
                # Solution: tile in depth
                # =============================================================
                x = _tiled_forward(dec, x, tile_depth=TILE_DEPTH)
            else:
                x = dec(x)

        return self.final(x)


print("V2 model (ResEncUNet3D + scSE) defined")
print(f"  Encoder stage 0: ResBlocks tiled ({TILE_DEPTH} depth) to avoid OOM")
print(f"  Decoder last stage: V2ConvBlock tiled ({TILE_DEPTH} depth) to avoid OOM")

In [ ]:
# =============================================================================
# NORMALIZATION FUNCTIONS - EACH MODEL USES ITS OWN
# =============================================================================

def robust_zscore_normalize(img, lower_percentile=0.5, upper_percentile=99.5):
    """
    V11 normalization: percentile clipping + Z-score.
    MUST match V11 training exactly.
    """
    p_low = np.percentile(img, lower_percentile)
    p_high = np.percentile(img, upper_percentile)
    img_clipped = np.clip(img, p_low, p_high)
    mean = img_clipped.mean()
    std = img_clipped.std()
    return ((img_clipped - mean) / (std + 1e-8)).astype(np.float32)


def simple_zscore_normalize(volume):
    """
    V2 normalization: simple Z-score.
    MUST match V2 training exactly.
    """
    return ((volume.astype(np.float32) - volume.mean()) / (volume.std() + 1e-8)).astype(np.float32)


print("Normalization functions:")
print("  V11: robust_zscore_normalize (percentile clipping 0.5-99.5% + Z-score)")
print("  V2:  simple_zscore_normalize (mean/std Z-score)")

In [ ]:
# =============================================================================
# INFERENCE FUNCTIONS
# Single GPU, batch_size=1, no TTA for maximum robustness.
# =============================================================================

def create_gaussian_weight(patch_size, sigma=0.125):
    """Gaussian blending weights for sliding window."""
    d, h, w = patch_size
    gz = np.exp(-0.5 * ((np.arange(d) - d / 2) / (d * sigma)) ** 2)
    gy = np.exp(-0.5 * ((np.arange(h) - h / 2) / (h * sigma)) ** 2)
    gx = np.exp(-0.5 * ((np.arange(w) - w / 2) / (w * sigma)) ** 2)
    return (gz[:, None, None] * gy[None, :, None] * gx[None, None, :]).astype(np.float32)


def get_patch_positions(volume_shape, patch_size, overlap=0.5):
    """Generate all patch start positions for sliding window."""
    D, H, W = volume_shape
    pd, ph, pw = patch_size
    sd = max(1, int(pd * (1 - overlap)))
    sh = max(1, int(ph * (1 - overlap)))
    sw = max(1, int(pw * (1 - overlap)))

    def _positions(total, patch, stride):
        if total <= patch:
            return [0]
        pos = list(range(0, total - patch + 1, stride))
        if (total - patch) not in pos:
            pos.append(total - patch)
        return pos

    z_pos = _positions(D, pd, sd)
    y_pos = _positions(H, ph, sh)
    x_pos = _positions(W, pw, sw)

    positions = []
    for z in z_pos:
        for y in y_pos:
            for x in x_pos:
                positions.append((z, y, x))
    return positions


@torch.no_grad()
def sliding_window_inference(model, volume_normalized, patch_size, overlap=0.5,
                             device="cuda", batch_size=1):
    """
    Sliding window inference on an ALREADY-NORMALIZED volume.
    Returns probability map (after sigmoid).
    
    Always batch_size=1 for safety on T4.
    """
    model.eval()
    D, H, W = volume_normalized.shape
    pd, ph, pw = patch_size

    # Pad if volume is smaller than patch
    pad_d = max(0, pd - D)
    pad_h = max(0, ph - H)
    pad_w = max(0, pw - W)
    if pad_d > 0 or pad_h > 0 or pad_w > 0:
        volume_normalized = np.pad(
            volume_normalized, ((0, pad_d), (0, pad_h), (0, pad_w)), mode='reflect'
        )
        D, H, W = volume_normalized.shape
        print(f"    Padded to: ({D}, {H}, {W})")

    pred_sum = np.zeros((D, H, W), dtype=np.float32)
    weight_sum = np.zeros((D, H, W), dtype=np.float32)
    gauss = create_gaussian_weight(patch_size)

    positions = get_patch_positions((D, H, W), patch_size, overlap)
    print(f"    Patches: {len(positions)}, Batch size: {batch_size}")

    for idx, (z, y, x) in enumerate(tqdm(positions, desc="  Infer", leave=False)):
        patch = volume_normalized[z:z + pd, y:y + ph, x:x + pw]

        # Pad if at edge
        actual_shape = patch.shape
        if actual_shape != (pd, ph, pw):
            p = [
                (0, max(0, pd - actual_shape[0])),
                (0, max(0, ph - actual_shape[1])),
                (0, max(0, pw - actual_shape[2])),
            ]
            patch = np.pad(patch, p, mode='reflect')

        batch_tensor = torch.from_numpy(patch[None, None]).to(device).half()

        with torch.cuda.amp.autocast(dtype=torch.float16):
            pred = torch.sigmoid(model(batch_tensor))

        # Use explicit indexing [0, 0] to remove batch & channel dims safely
        # (don't use .squeeze() — it would break if a spatial dim happens to be 1)
        pred_np = pred[0, 0].float().cpu().numpy()

        pred_sum[z:z + pd, y:y + ph, x:x + pw] += pred_np * gauss
        weight_sum[z:z + pd, y:y + ph, x:x + pw] += gauss

        del batch_tensor, pred, pred_np, patch

        # Periodic cache clear
        if idx % 20 == 0:
            torch.cuda.empty_cache()

    result = pred_sum / np.maximum(weight_sum, 1e-8)

    # Remove padding
    orig_D = D - pad_d if pad_d > 0 else D
    orig_H = H - pad_h if pad_h > 0 else H
    orig_W = W - pad_w if pad_w > 0 else W
    result = result[:orig_D, :orig_H, :orig_W]

    return result


print("Inference functions ready")
print("  Sliding window: batch=1, Gaussian blending, no TTA")

In [ ]:
# =============================================================================
# POST-PROCESSING (Improved based on Kaggle Discussion Analysis)
#
# Key insights from competition discussion:
# 1. betti-1 errors (holes) cost ~0.05 PER HOLE on LB — aggressive hole filling
# 2. Ignore mask can create artificial holes — thicker/smoother predictions help
# 3. VOI includes background — ensemble averaging smooths boundaries
# 4. Topology-safe ops: never merge distinct components
#
# Pipeline: 3D closing → 3D hole fill → slicewise morphology → cleanup
# =============================================================================

def count_components(mask):
    struct = generate_binary_structure(3, 3)
    _, n = label(mask, structure=struct)
    return n


def remove_small_components(mask, min_size=50):
    struct = generate_binary_structure(3, 3)
    labeled, n = label(mask, structure=struct)
    if n == 0:
        return mask
    sizes = np.bincount(labeled.ravel())
    small = sizes < min_size
    small[0] = False
    result = mask.copy()
    result[small[labeled]] = 0
    return result


def topology_safe_operation(mask, operation_func, name="op"):
    """Apply operation only if it doesn't reduce component count (no merging)."""
    n_before = count_components(mask)
    result = operation_func(mask)
    n_after = count_components(result)
    if n_after < n_before:
        print(f"    [REVERT] {name}: would merge {n_before}->{n_after} components")
        return mask
    return result


def slicewise_hole_fill(mask):
    filled = mask.copy()
    for i in range(mask.shape[0]):
        filled[i] = binary_fill_holes(filled[i])
    for i in range(mask.shape[1]):
        filled[:, i, :] = binary_fill_holes(filled[:, i, :])
    for i in range(mask.shape[2]):
        filled[:, :, i] = binary_fill_holes(filled[:, :, i])
    return filled


def slicewise_morphology(mask, operation='close', iterations=1):
    result = mask.copy()
    struct_2d = generate_binary_structure(2, 1)
    for axis in range(3):
        for i in range(mask.shape[axis]):
            if axis == 0:
                slc = result[i]
            elif axis == 1:
                slc = result[:, i, :]
            else:
                slc = result[:, :, i]
            if operation == 'close':
                slc_new = binary_closing(slc, structure=struct_2d, iterations=iterations)
            elif operation == 'open':
                slc_new = binary_opening(slc, structure=struct_2d, iterations=iterations)
            else:
                slc_new = slc
            if axis == 0:
                result[i] = slc_new
            elif axis == 1:
                result[:, i, :] = slc_new
            else:
                result[:, :, i] = slc_new
    return result


def postprocess_ensemble(pred_prob, threshold=0.50, min_component_size=50, verbose=True):
    """
    Post-processing for 3-model ensemble output.
    
    Discussion-informed improvements:
    - 3D closing before slicewise: catches 3D topology errors slicewise misses
    - 3D hole filling: betti-1 errors cost ~0.05 per hole on LB
    - Topology-safe: revert any operation that merges distinct components
    """
    if verbose:
        print("Post-processing (discussion-informed):")
        print(f"  Input: min={pred_prob.min():.3f}, max={pred_prob.max():.3f}, mean={pred_prob.mean():.3f}")

    # 1. Threshold
    mask = (pred_prob > threshold).astype(np.uint8)
    if verbose:
        print(f"  1. Threshold ({threshold}): {mask.sum():,} voxels, FG={100 * mask.mean():.2f}%")

    if mask.sum() == 0:
        if verbose:
            print("  WARNING: Empty mask!")
        return mask

    # 2. Remove small components (noise)
    n_before = count_components(mask)
    mask = remove_small_components(mask, min_component_size)
    n_after = count_components(mask)
    if verbose:
        print(f"  2. Remove small (<{min_component_size}): {n_before}->{n_after} components")

    # 3. 3D binary closing (topology-safe) — bridges 1-voxel 3D gaps
    #    Uses 26-connectivity (full 3x3x3 cube) to catch diagonal gaps
    #    This is critical: slicewise closing misses 3D gaps between planes
    struct_3d_full = generate_binary_structure(3, 3)  # 26-connectivity
    mask = topology_safe_operation(
        mask,
        lambda m: binary_closing(m, structure=struct_3d_full, iterations=1).astype(np.uint8),
        "3D_closing_26conn"
    )
    if verbose:
        print(f"  3. 3D closing (26-conn): FG={100 * mask.mean():.2f}%")

    # 4. 3D hole filling (topology-safe) — fills enclosed 3D voids
    #    CRITICAL: Each hole costs ~0.05 on LB (betti-1 penalty)
    #    binary_fill_holes fills any void completely enclosed by foreground
    mask = topology_safe_operation(
        mask,
        lambda m: binary_fill_holes(m).astype(np.uint8),
        "3D_fill_holes"
    )
    if verbose:
        print(f"  4. 3D hole fill: FG={100 * mask.mean():.2f}%")

    # 5. Slicewise closing (topology-safe) — bridges 2D gaps within each plane
    mask = topology_safe_operation(
        mask, lambda m: slicewise_morphology(m, 'close', iterations=1), "slicewise_close"
    )
    if verbose:
        print(f"  5. Slicewise closing: FG={100 * mask.mean():.2f}%")

    # 6. Slicewise hole fill (topology-safe) — fills 2D holes per-slice
    mask = topology_safe_operation(mask, slicewise_hole_fill, "slicewise_hole_fill")
    if verbose:
        print(f"  6. Slicewise hole fill: FG={100 * mask.mean():.2f}%")

    # 7. Second 3D hole fill — catches any holes created by slicewise ops
    mask = topology_safe_operation(
        mask,
        lambda m: binary_fill_holes(m).astype(np.uint8),
        "3D_fill_holes_2"
    )
    if verbose:
        print(f"  7. 3D hole fill (2nd pass): FG={100 * mask.mean():.2f}%")

    # 8. Slicewise opening (topology-safe) — smooth rough edges
    mask = topology_safe_operation(
        mask, lambda m: slicewise_morphology(m, 'open', iterations=1), "slicewise_open"
    )
    if verbose:
        print(f"  8. Slicewise opening: FG={100 * mask.mean():.2f}%")

    # 9. Final cleanup
    mask = remove_small_components(mask, min_component_size)
    n_final = count_components(mask)
    if verbose:
        print(f"  Final: {n_final} components, {mask.sum():,} voxels, FG={100 * mask.mean():.2f}%")

    return mask


print("Post-processing ready (discussion-informed)")
print("  NEW: 3D closing (26-conn) + 3D hole fill before slicewise ops")
print("  NEW: Second 3D hole fill pass after slicewise ops")
print("  Each hole costs ~0.05 on LB → aggressive filling")

In [ ]:
# =============================================================================
# MODEL LOADING — 3 MODELS + OPTIONAL CHECKPOINT AVERAGING
#
# Models are loaded ON DEMAND and offloaded to CPU after use.
# Only one model is on GPU at a time during inference.
# NO DataParallel — single GPU, batch_size=1 always.
#
# V11 fold 0: optional SWA (weight average of multiple checkpoints)
# V11.1 fold 1: single checkpoint (new architecture)
# V2: single checkpoint
# =============================================================================

def average_checkpoint_weights(checkpoint_paths, device='cpu'):
    """
    Average weights from multiple checkpoints (SWA-like).
    All checkpoints must be from the same architecture.
    Returns averaged state_dict or None if no valid checkpoints.
    """
    state_dicts = []
    for path in checkpoint_paths:
        path = Path(path)
        if not path.exists():
            print(f"  Warning: {path} not found, skipping")
            continue
        ckpt = torch.load(path, map_location=device, weights_only=False)
        state = ckpt.get('model_state_dict', ckpt)
        state = {k.replace('_orig_mod.', '').replace('module.', ''): v
                 for k, v in state.items()}
        state_dicts.append(state)
        epoch = ckpt.get('epoch', '?')
        dice = ckpt.get('best_dice', 0)
        print(f"  Loaded: {path.name} (epoch={epoch}, dice={dice:.4f})")
        del ckpt

    if len(state_dicts) == 0:
        return None
    if len(state_dicts) == 1:
        return state_dicts[0]

    # Average all weights
    avg_state = {}
    for key in state_dicts[0].keys():
        tensors = [sd[key].float() for sd in state_dicts if key in sd]
        if tensors:
            avg_state[key] = torch.mean(torch.stack(tensors), dim=0)

    print(f"  Averaged {len(state_dicts)} checkpoints (SWA)")
    return avg_state


def load_v11_model(checkpoint_path, features, n_blocks, extra_checkpoints=None, name="V11"):
    """Load V11 TopoPreservingUNet3D from checkpoint(s) to CPU."""
    print(f"\nLoading {name} model:")
    model = TopoPreservingUNet3D(features=features, n_blocks=n_blocks)
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"  Architecture: features={features}, blocks={n_blocks}")
    print(f"  Parameters: {n_params:.1f}M")

    # Check if we should use checkpoint averaging (SWA)
    all_paths = [checkpoint_path]
    if extra_checkpoints:
        all_paths.extend(extra_checkpoints)
    existing_paths = [p for p in all_paths if Path(p).exists()]

    if len(existing_paths) == 0:
        raise FileNotFoundError(f"{name} checkpoint not found: {checkpoint_path}")

    if len(existing_paths) > 1:
        # SWA: average weights from multiple checkpoints
        print(f"  Using SWA with {len(existing_paths)} checkpoints:")
        avg_state = average_checkpoint_weights(existing_paths)
        model.load_state_dict(avg_state, strict=False)
    else:
        # Single checkpoint
        path = existing_paths[0]
        print(f"  Loading: {path}")
        ckpt = torch.load(path, map_location="cpu", weights_only=False)
        state = ckpt.get('model_state_dict', ckpt)
        state = {k.replace('_orig_mod.', '').replace('module.', ''): v
                 for k, v in state.items()}
        model.load_state_dict(state, strict=False)
        print(f"  Epoch: {ckpt.get('epoch', '?')}, Best Dice: {ckpt.get('best_dice', 0):.4f}")
        del ckpt, state

    gc.collect()
    model = model.half().eval()
    print(f"  {name} ready on CPU")
    return model


def load_v2_model(checkpoint_path, features, n_blocks):
    """Load V2 ResEncUNet3D from checkpoint to CPU."""
    print(f"\nLoading V2 model:")
    if not Path(checkpoint_path).exists():
        raise FileNotFoundError(f"V2 checkpoint not found: {checkpoint_path}")

    model = ResEncUNet3D(
        features=features, n_blocks=n_blocks,
        use_scse=True, use_deep_supervision=True,
    )
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"  Architecture: features={features}, blocks={n_blocks}")
    print(f"  Parameters: {n_params:.1f}M")

    ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    state_dict = ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt
    cleaned = {k.replace('module.', '').replace('_orig_mod.', ''): v
               for k, v in state_dict.items()}
    model_keys = set(model.state_dict().keys())
    cleaned = {k: v for k, v in cleaned.items() if k in model_keys}
    missing, unexpected = model.load_state_dict(cleaned, strict=False)
    if missing:
        print(f"  Missing keys (expected for DS heads): {len(missing)}")
    del ckpt, state_dict, cleaned
    gc.collect()
    model = model.half().eval()
    print(f"  V2 ready on CPU")
    return model


def move_to_gpu(model):
    """Move model to GPU. No DataParallel — single GPU only."""
    model = model.cuda()
    torch.cuda.empty_cache()
    return model


def offload_to_cpu(model):
    """Offload model from GPU to CPU to free VRAM."""
    model = model.cpu()
    torch.cuda.empty_cache()
    gc.collect()
    return model


# =========================================================================
# LOAD ALL 3 MODELS TO CPU
# =========================================================================
print("=" * 70)
print("Loading 3 models to CPU...")
print("=" * 70)

# Model 1: V11 Fold 0 (old architecture, optional SWA)
model_v11_f0 = load_v11_model(
    cfg.V11_F0_CHECKPOINT,
    cfg.V11_F0_FEATURES,
    cfg.V11_F0_N_BLOCKS,
    extra_checkpoints=cfg.V11_F0_EXTRA_CHECKPOINTS,
    name="V11-F0"
)

# Model 2: V11.1 Fold 1 (new 512ch architecture)
model_v11_f1 = load_v11_model(
    cfg.V11_F1_CHECKPOINT,
    cfg.V11_F1_FEATURES,
    cfg.V11_F1_N_BLOCKS,
    name="V11.1-F1"
)

# Model 3: V2 (ResEncUNet3D + scSE)
model_v2 = load_v2_model(
    cfg.V2_CHECKPOINT,
    cfg.V2_FEATURES,
    cfg.V2_N_BLOCKS
)

print("\n" + "=" * 70)
print("All 3 models loaded to CPU (will move to GPU one at a time)")
print(f"  V11-F0:   TopoPreservingUNet3D [320,320] | Weight: {cfg.V11_F0_WEIGHT}")
print(f"  V11.1-F1: TopoPreservingUNet3D [512,512] | Weight: {cfg.V11_F1_WEIGHT}")
print(f"  V2:       ResEncUNet3D + scSE            | Weight: {cfg.V2_WEIGHT}")
print("=" * 70)

In [ ]:
# =============================================================================
# 3-MODEL ENSEMBLE INFERENCE PIPELINE
#
# For each test volume:
# 1. V11-F0 on GPU → infer (robust Z-score) → offload
# 2. V11.1-F1 on GPU → infer (robust Z-score) → offload
# 3. V2 on GPU → infer (simple Z-score) → offload
# 4. Weighted average of 3 probability maps
#
# Fallback: if any model fails, use remaining models (re-normalized weights)
# =============================================================================

def gpu_status():
    """Print current GPU memory usage."""
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        free = torch.cuda.mem_get_info()[0] / 1e9
        print(f"  GPU mem: {alloc:.1f}G alloc, {reserved:.1f}G reserved, {free:.1f}G free")


def run_single_model_inference(model, volume_normalized, name, cfg):
    """
    Run inference for a single model.
    Model is moved to GPU, inference runs, then model is offloaded.
    Returns (probability_map, model) or (None, model) on failure.
    """
    patch_size = cfg.PATCH_SIZE
    overlap = cfg.OVERLAP
    device = cfg.DEVICE

    try:
        print(f"\n  [{name}] Moving to GPU...")
        model = move_to_gpu(model)
        gpu_status()

        print(f"  [{name}] Running sliding window inference (batch=1)...")
        prob = sliding_window_inference(
            model, volume_normalized, patch_size, overlap,
            device=device, batch_size=1,
        )
        print(f"  [{name}] Prob: min={prob.min():.4f}, max={prob.max():.4f}, mean={prob.mean():.4f}")

        print(f"  [{name}] Offloading to CPU...")
        model = offload_to_cpu(model)
        gpu_status()

        return prob, model

    except Exception as e:
        print(f"\n  [{name}] ERROR during inference: {type(e).__name__}: {e}")
        traceback.print_exc()
        try:
            model = model.cpu()
        except:
            pass
        torch.cuda.empty_cache()
        gc.collect()
        return None, model


def run_3model_ensemble(volume_raw, model_v11_f0, model_v11_f1, model_v2, cfg):
    """
    Run 3-model ensemble inference:
    1. V11-F0 on GPU → infer → offload
    2. V11.1-F1 on GPU → infer → offload
    3. V2 on GPU → infer → offload
    4. Weighted average of probability maps (with fallback)
    """
    print("\n" + "=" * 60)
    print("3-MODEL ENSEMBLE INFERENCE")
    print(f"  Volume shape: {volume_raw.shape}")
    print(f"  Weights: V11-F0={cfg.V11_F0_WEIGHT}, V11.1-F1={cfg.V11_F1_WEIGHT}, V2={cfg.V2_WEIGHT}")
    print("=" * 60)

    # =====================================================================
    # Step 1: V11 Fold 0 (robust Z-score normalization)
    # =====================================================================
    print("\n" + "-" * 50)
    print("[V11-F0] TopoPreservingUNet3D [320,320] — Fold 0")
    print("-" * 50)

    vol_robust = robust_zscore_normalize(volume_raw)
    print(f"  Normalized (robust Z-score): mean={vol_robust.mean():.4f}, std={vol_robust.std():.4f}")

    prob_v11_f0, model_v11_f0 = run_single_model_inference(model_v11_f0, vol_robust, "V11-F0", cfg)

    # =====================================================================
    # Step 2: V11.1 Fold 1 (same robust Z-score — same normalization as V11)
    # =====================================================================
    print("\n" + "-" * 50)
    print("[V11.1-F1] TopoPreservingUNet3D [512,512] — Fold 1")
    print("-" * 50)

    # Reuse vol_robust — same normalization method
    print(f"  Reusing robust Z-score normalized volume")

    prob_v11_f1, model_v11_f1 = run_single_model_inference(model_v11_f1, vol_robust, "V11.1-F1", cfg)
    del vol_robust
    gc.collect()

    # =====================================================================
    # Step 3: V2 (simple Z-score normalization — different from V11!)
    # =====================================================================
    print("\n" + "-" * 50)
    print("[V2] ResEncUNet3D + scSE")
    print("-" * 50)

    vol_simple = simple_zscore_normalize(volume_raw)
    print(f"  Normalized (simple Z-score): mean={vol_simple.mean():.4f}, std={vol_simple.std():.4f}")

    prob_v2, model_v2 = run_single_model_inference(model_v2, vol_simple, "V2", cfg)
    del vol_simple
    gc.collect()
    torch.cuda.empty_cache()

    # =====================================================================
    # Step 4: Weighted ensemble (with fallback)
    # =====================================================================
    print("\n" + "-" * 50)
    print("ENSEMBLE COMBINATION")
    print("-" * 50)

    # Collect successful predictions with their weights
    predictions = []
    weights = []
    names = []

    if prob_v11_f0 is not None:
        predictions.append(prob_v11_f0)
        weights.append(cfg.V11_F0_WEIGHT)
        names.append("V11-F0")
    if prob_v11_f1 is not None:
        predictions.append(prob_v11_f1)
        weights.append(cfg.V11_F1_WEIGHT)
        names.append("V11.1-F1")
    if prob_v2 is not None:
        predictions.append(prob_v2)
        weights.append(cfg.V2_WEIGHT)
        names.append("V2")

    if len(predictions) == 0:
        print(f"  CRITICAL: All models failed! Returning empty prediction.")
        prob_ensemble = np.zeros(volume_raw.shape, dtype=np.float32)
    elif len(predictions) == 3:
        # All models succeeded — weighted average
        w_total = sum(weights)
        prob_ensemble = sum(w * p for w, p in zip(weights, predictions)) / w_total
        print(f"  Mode: 3-model weighted average")
        print(f"    {' + '.join(f'{w:.2f}*{n}' for w, n in zip(weights, names))}")
    else:
        # Partial success — re-normalize weights
        w_total = sum(weights)
        normalized_weights = [w / w_total for w in weights]
        prob_ensemble = sum(w * p for w, p in zip(normalized_weights, predictions))
        failed = set(["V11-F0", "V11.1-F1", "V2"]) - set(names)
        print(f"  WARNING: {failed} failed, using {names} only")
        print(f"    Re-normalized: {' + '.join(f'{w:.2f}*{n}' for w, n in zip(normalized_weights, names))}")

    # Agreement statistics (if we have multiple predictions)
    if len(predictions) >= 2:
        binary_preds = [(p > 0.5) for p in predictions]
        all_agree_fg = binary_preds[0]
        all_agree_bg = ~binary_preds[0]
        for bp in binary_preds[1:]:
            all_agree_fg = all_agree_fg & bp
            all_agree_bg = all_agree_bg & ~bp
        n_total = predictions[0].size
        n_agree = all_agree_fg.sum() + all_agree_bg.sum()
        n_disagree = n_total - n_agree
        print(f"  Agreement: {100 * n_agree / n_total:.1f}% | Disagree: {100 * n_disagree / n_total:.1f}%")

    print(f"  Ensemble prob: min={prob_ensemble.min():.4f}, max={prob_ensemble.max():.4f}, mean={prob_ensemble.mean():.4f}")

    del prob_v11_f0, prob_v11_f1, prob_v2, predictions
    gc.collect()

    return prob_ensemble, model_v11_f0, model_v11_f1, model_v2


print("3-model ensemble pipeline ready")
print("  V11-F0 (robust Z-score) → V11.1-F1 (robust Z-score) → V2 (simple Z-score)")
print("  Fallback: partial ensemble with re-normalized weights if any model fails")

In [ ]:
# =============================================================================
# RUN 3-MODEL ENSEMBLE ON ALL TEST VOLUMES
# =============================================================================

import time

# Find test volumes
test_files = sorted(cfg.TEST_ROOT.glob("*.tif")) + sorted(cfg.TEST_ROOT.glob("*.tiff"))
print(f"Found {len(test_files)} test volumes")
for f in test_files:
    print(f"  {f.name}")

if len(test_files) == 0:
    print("WARNING: No test volumes found! Creating empty submission.")

# Create output directory
mask_dir = cfg.OUTPUT_DIR / "masks"
mask_dir.mkdir(exist_ok=True, parents=True)

total_start = time.time()

for vol_idx, vol_path in enumerate(test_files):
    vol_id = vol_path.stem
    vol_start = time.time()

    print(f"\n{'=' * 70}")
    print(f"Processing [{vol_idx+1}/{len(test_files)}]: {vol_id}")
    print(f"{'=' * 70}")

    try:
        # Clear GPU memory before each volume
        torch.cuda.empty_cache()
        gc.collect()
        gpu_status()

        # Load volume
        print(f"Loading volume: {vol_path}")
        volume = tifffile.imread(str(vol_path)).astype(np.float32)
        original_shape = volume.shape
        print(f"  Shape: {original_shape}")
        print(f"  Dtype: {volume.dtype}, Range: [{volume.min():.1f}, {volume.max():.1f}]")
        print(f"  Size: {volume.nbytes / 1e9:.2f} GB")

        # Sanity check
        if volume.ndim != 3:
            print(f"  ERROR: Expected 3D volume, got {volume.ndim}D! Skipping.")
            empty_mask = np.zeros(original_shape[:3] if volume.ndim >= 3 else (1, 1, 1), dtype=np.uint8)
            mask_path = mask_dir / f"{vol_id}.tif"
            tifffile.imwrite(str(mask_path), empty_mask, compression=None)
            continue

        if volume.size == 0:
            print(f"  ERROR: Empty volume! Skipping.")
            mask_path = mask_dir / f"{vol_id}.tif"
            tifffile.imwrite(str(mask_path), np.zeros((1, 1, 1), dtype=np.uint8), compression=None)
            continue

        # Run 3-model ensemble inference
        prob_ensemble, model_v11_f0, model_v11_f1, model_v2 = run_3model_ensemble(
            volume, model_v11_f0, model_v11_f1, model_v2, cfg
        )

        del volume
        gc.collect()

        # Post-processing (discussion-informed: aggressive hole filling)
        print()
        pred_mask = postprocess_ensemble(
            prob_ensemble,
            threshold=cfg.THRESHOLD,
            min_component_size=cfg.MIN_COMPONENT_SIZE,
            verbose=True,
        )

        # Verify shape
        if pred_mask.shape != original_shape:
            print(f"  WARNING: Shape mismatch: {pred_mask.shape} vs {original_shape}")
            pred_fixed = np.zeros(original_shape, dtype=np.uint8)
            slices = tuple(slice(0, min(s1, s2)) for s1, s2 in zip(pred_mask.shape, original_shape))
            pred_fixed[slices] = pred_mask[slices]
            pred_mask = pred_fixed

        # Save TIFF (no compression for speed)
        mask_path = mask_dir / f"{vol_id}.tif"
        tifffile.imwrite(str(mask_path), pred_mask.astype(np.uint8), compression=None)
        file_size = mask_path.stat().st_size / 1e6
        vol_time = time.time() - vol_start
        print(f"\nSaved: {mask_path.name} ({file_size:.2f} MB) in {vol_time:.1f}s")

        del prob_ensemble, pred_mask
        gc.collect()
        torch.cuda.empty_cache()

    except Exception as e:
        print(f"\n{'!' * 70}")
        print(f"CRITICAL ERROR processing {vol_id}: {type(e).__name__}: {e}")
        traceback.print_exc()
        print(f"{'!' * 70}")

        # Save empty mask so submission still works
        try:
            mask_path = mask_dir / f"{vol_id}.tif"
            with tifffile.TiffFile(str(vol_path)) as tif:
                shape = tif.pages[0].shape if len(tif.pages) == 1 else (len(tif.pages),) + tif.pages[0].shape
            empty_mask = np.zeros(shape, dtype=np.uint8)
            tifffile.imwrite(str(mask_path), empty_mask, compression=None)
            print(f"  Saved EMPTY mask for {vol_id} (shape={shape})")
        except Exception as e2:
            print(f"  Failed to save empty mask: {e2}")

        # Try to recover GPU state
        for m in [model_v11_f0, model_v11_f1, model_v2]:
            try:
                if str(next(m.parameters()).device) != 'cpu':
                    m = m.cpu()
            except:
                pass
        torch.cuda.empty_cache()
        gc.collect()

total_time = time.time() - total_start
print(f"\n{'=' * 70}")
print(f"3-MODEL ENSEMBLE INFERENCE COMPLETE")
print(f"  Models: V11-F0 ({cfg.V11_F0_WEIGHT}) + V11.1-F1 ({cfg.V11_F1_WEIGHT}) + V2 ({cfg.V2_WEIGHT})")
print(f"  Volumes processed: {len(test_files)}")
print(f"  Total time: {total_time:.1f}s ({total_time/60:.1f}m)")
print(f"{'=' * 70}")

In [ ]:
# =============================================================================
# CREATE SUBMISSION ZIP
# =============================================================================

submission_zip = cfg.OUTPUT_DIR / "submission.zip"

mask_files = sorted(mask_dir.glob("*.tif"))
print(f"Mask files to include: {len(mask_files)}")

if len(mask_files) == 0:
    print("WARNING: No mask files found! Creating empty submission.")
    with zipfile.ZipFile(submission_zip, 'w') as zf:
        pass
else:
    with zipfile.ZipFile(submission_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
        for tif_path in mask_files:
            zf.write(tif_path, tif_path.name)
            print(f"  Added: {tif_path.name} ({tif_path.stat().st_size / 1e6:.2f} MB)")

    # Verify
    print(f"\nSubmission: {submission_zip}")
    print(f"  Size: {submission_zip.stat().st_size / 1e6:.2f} MB")
    with zipfile.ZipFile(submission_zip, 'r') as zf:
        for info in zf.infolist():
            print(f"  {info.filename}: {info.file_size / 1e6:.2f} MB (compressed: {info.compress_size / 1e6:.2f} MB)")

print(f"\n>>> Ready to submit: {submission_zip}")
print(f"\n3-Model Ensemble:")
print(f"  V11-F0  ({cfg.V11_F0_WEIGHT:.0%}): TopoPreservingUNet3D [320,320] — Fold 0, Dice 0.5793")
print(f"  V11.1-F1 ({cfg.V11_F1_WEIGHT:.0%}): TopoPreservingUNet3D [512,512] — Fold 1, Dice 0.5596")
print(f"  V2      ({cfg.V2_WEIGHT:.0%}): ResEncUNet3D + scSE — different architecture")
print(f"\nPost-processing: 3D closing + 3D hole fill + slicewise morphology")
print(f"Settings: single GPU, batch=1, no DataParallel")
print(f"OOM fixes: tiled Conv3d in V11 SurfaceRefinement + V2 encoder/decoder")